# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [1]:
from typing_extensions import TypedDict

class NewsroomState(TypedDict):
    stories: dict   # keyed by story_id (slug), enriched in place by each agent

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [2]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [3]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 8
    40.6 vel | giant-trees-have-no-trouble-pumping-water-to-top-branches-new-research
    37.6 vel | agentic-coding-notes-from-galapagos-island
    35.5 vel | performance-per-dollar-is-getting-faster-and-cheaper
    31.3 vel | leanstral-1-5-proof-abundance-for-all
    27.0 vel | the-bottleneck-might-be-the-air-in-the-room
    16.8 vel | synthesis-is-harder-than-analysis
     7.1 vel | 2026-unslop-ai-written-fiction-contest-results
     0.7 vel | mir-books-books-from-the-soviet-era


---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [4]:
import ollama, subprocess, time

# Health-check: Ollama must be running for Agent 2 synthesis
try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama already running


In [5]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [6]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : giant-trees-have-no-trouble-pumping-water-to-top-branches-new-research
Trafiltura Success ✅  In Loading Content :3654 characters
  [background] gathering snippets for: Giant trees have no trouble pumping water to top branches: new research
  [bg] DDG returned 2 snippets
  [bg] Wikipedia search returns: 1099 chars
  [background] 3 snippets, synthesizing (content-anchored)...
  [synth] 3 snippets, 1877 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 607 chars
Result After Background Syntezing is : Tropical forests are home to some of the world's tallest trees, including Dipterocarp species, which dominate Asian rainforests. These towering trees can reach heights of over 80 meters, but conventional scientific theory suggests that their water transport systems should be severely limited by gravity and vessel length, making them more vulnerable to drought. However, new research has challenged this idea, revealing that giant tropical trees have evolved intricate adap

/Users/deepakrathore/Documents/Agentic-AI/Examples/NewsStudio/multi-agent-env/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/deepakrathore/Documents/Agentic-AI/Examples/NewsStudio/multi-agent-env/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


  [bg] Wikipedia search returns: empty (skipped)
  [background] 4 snippets, synthesizing (content-anchored)...
  [synth] 4 snippets, 977 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 659 chars
Result After Background Syntezing is : Mathematicians and computer scientists have developed various calculi over the years, including the lambda calculus, relational calculus, predicate calculus, and sequent calculus. However, when referring to "calculus" without modification, it typically refers to one of two related calculi: differential calculus or integral calculus. Differential calculus is about calculating the slope of a function at a given point, while integral calculus is concerned with finding the area under a graph over a particular interval. These two calculi are often taught in sequence, with differential calculus (Cal 1) coming first and followed by integral calculus (Cal 2).
  [bg] background: 659 chars
  Agent 2 Ends for: Synthesis is harder than analysis
Agent 2 Starts For 

In [7]:
# Inspect Agent 2 output
for sid, story in call2["stories"].items():
    content_len  = len(story.get("content",  ""))
    bg_len       = len(story.get("background",""))
    bg_status    = f"{bg_len} chars" if bg_len else "EMPTY"
    print(f"{story['velocity']:>6.1f} vel | content {content_len:>5}c | bg {bg_status:<12} | {story['title'][:55]}")

  40.6 vel | content  3654c | bg 607 chars    | Giant trees have no trouble pumping water to top branch
  37.6 vel | content  6000c | bg 642 chars    | Agentic coding notes from Galapagos Island
  35.5 vel | content  6000c | bg 433 chars    | Performance per dollar is getting faster and cheaper
  31.3 vel | content  6000c | bg 708 chars    | Leanstral 1.5: Proof abundance for all
  27.0 vel | content  3437c | bg 623 chars    | The bottleneck might be the air in the room
  16.8 vel | content  6000c | bg 659 chars    | Synthesis is harder than analysis
   7.1 vel | content  6000c | bg 414 chars    | 2026 Unslop AI-Written Fiction Contest Results
   0.7 vel | content  2977c | bg 475 chars    | Mir Books – Books from the Soviet Era


---
## Agent 3: Fact Checker
Scores each story's credibility on a **-1 to +1 scale**.
Zero is the natural boundary — negative = discard, positive = keep.

| Signal | Base Weight | How |
|--------|-------------|-----|
| `source_score` | 20% | Domain trust map (32 domains, 0.0 to +0.95) |
| `llm_credibility_check` | 60% | gpt-oss-120b → REAL +0.9 / OPINION +0.1 / SPAM -0.7 |
| `cross_verify` | 20% | Exa → DDG → confirms or contradicts via second source |

**Dynamic reweighting** — weights shift when cross-verify fires:
- Contradiction detected → llm 60%→30%, verify 20%→50%
- Confirmation detected  → llm 60%→50%, verify 20%→30%
- Neutral (not found)    → standard weights unchanged

**Key design decisions:**
- `-1 to +1` range — negative range earned by SPAM or credible contradiction
- Threshold = 0.0 — zero is the natural semantic boundary
- SPAM → -0.7 (only genuine negative LLM signal)
- Empty response / Groq failure → 0.0 neutral (never discard on crash)
- Thin content < 500 chars → 0.0 neutral
- Soft discard — story marked `discarded=True`, never deleted (audit trail)
- Cross-verify: Exa (semantic, HN-aware) → DDG fallback → neutral
- Contradiction check uses `compound-mini` (separate quota from 120b)

In [8]:
import time                                    # ← add this line
from agents.agent3 import (
    source_score,
    llm_credibility_check,
    check_credibility,
    cross_verify,
)
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [9]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag    = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        regime  = story.get("_cred_regime", "?")
        vby     = story.get("verified_by", "NONE")
        contra  = "❌ contradicted" if story.get("contradicted") else ""
        print(f"{story['credibility_score']:+.2f} {flag} "
              f"[{regime}] verified={vby} {contra}| {story['title'][:45]}")
        time.sleep(1)
    return {"stories": stories}

In [10]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

── CREDIBILITY RESULTS ─────────────────────────────
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'Giant trees have no trouble pumping wate' → 'REAL' → 0.9
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00×0.2 + llm=0.90×0.6 + verify=0.00×0.2 = 0.54
+0.54 ✅ KEEP [neutral] verified=NONE | Giant trees have no trouble pumping water to 
  [llm_cred_check DEBUG] raw='OPINION'
  [llm_cred_check] 'Agentic coding notes from Galapagos Isla' → 'OPINION' → 0.1
  [verify] contradiction check → 'UNRELATED'
  contradict claim[verify] ✅ confirmed by github.com (trust=0.95)
  [cred] regime=confirmation | src=0.70×0.2 + llm=0.10×0.5 + verify=0.80×0.3 = 0.43
+0.43 ✅ KEEP [confirmation] verified=github.com | Agentic coding notes from Galapagos Island
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'Performance per dollar is getting faster' → 'REAL' → 0.9
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=

In [13]:
for sid, story in call3["stories"].items():
    print(sid)
    print(story)

giant-trees-have-no-trouble-pumping-water-to-top-branches-new-research
{'title': 'Giant trees have no trouble pumping water to top branches: new research', 'url': 'https://news.exeter.ac.uk/faculty-of-environment-science-and-economy/giant-trees-have-no-trouble-pumping-water-to-top-branches/', 'source': 'hackernews', 'category': 'technology', 'engagement': 257, 'velocity': 40.6, 'content': 'Giant trees have no trouble pumping water to top branches\nThe world’s tallest tropical trees have no trouble pumping water to their topmost branches, new research reveals.\nConventional scientific theory suggests that as trees grow, it becomes harder to transport water from roots to leaves – limiting growth and making trees more vulnerable to drought.\nBut the new study – led by the University of Exeter and Cardiff University and published in the journal Science – finds that adjustments to water transport inside giant Dipterocarp trees “fully compensated” for the challenges of drawing water to the t

In [16]:
# ── INSPECT: full story details after all 3 agents ──────────────────
print(f"\n{'FLAG':<4} {'SCORE':>6} {'VEL':>6} {'CONTENT':>8} "
      f"{'BG':>5} {'REGIME':<14} {'VERIFIED_BY':<20} TITLE")
print("-" * 115)



for sid, story in call3["stories"].items():
    flag    = "🗑️" if story.get("discarded") else "✅"
    score   = story.get("credibility_score", 0)
    vel     = story.get("velocity", 0)
    clen    = len(story.get("content", ""))
    bglen   = len(story.get("background", ""))
    regime  = story.get("_cred_regime", "?")
    vby     = story.get("verified_by", "NONE")
    contra  = "❌" if story.get("contradicted") else ""
    #weights = story.get("_weights_used", "?")
    title   = story.get("title", "")[:35]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {clen:>6}c  "
          f"{bglen:>4}c  {regime:<14} {contra}{vby:<20}{title}")


FLAG  SCORE    VEL  CONTENT    BG REGIME         VERIFIED_BY          TITLE
-------------------------------------------------------------------------------------------------------------------
✅  +0.54    40.6    3654c   607c  neutral        NONE                Giant trees have no trouble pumping
✅  +0.43    37.6    6000c   642c  confirmation   github.com          Agentic coding notes from Galapagos
✅  +0.54    35.5    6000c   433c  neutral        NONE                Performance per dollar is getting f
✅  +0.54    31.3    6000c   708c  neutral        NONE                Leanstral 1.5: Proof abundance for 
✅  +0.06    27.0    3437c   623c  neutral        NONE                The bottleneck might be the air in 
✅  +0.06    16.8    6000c   659c  neutral        NONE                Synthesis is harder than analysis
✅  +0.54     7.1    6000c   414c  neutral        NONE                2026 Unslop AI-Written Fiction Cont
✅  +0.54     0.7    2977c   475c  neutral        NONE                Mir B

### Save story : till agent 3 (quality data progress saved to disk) so even if some changes comes in agent 1 to agent 3 , agent 4+ code developement will not be affect

In [12]:

# Save to disk — skip re-processing next time
#from agent_tools.story_cache import save_stories
#save_stories(call3["stories"])

# TO track log of particular function after a hit of particular no. of times pop-up will come
